## Step 1: Extract Text from a PDF

First, we need to install a library to handle PDF files. `pdfplumber` is a good choice for extracting text.

**Note:** You will need to upload your PDF file to the Colab environment (e.g., by dragging it into the files tab on the left sidebar) before you can extract text from it. For demonstration, I'll use a placeholder filename `your_document.pdf`.

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor, XGBClassifier



In [ ]:
pip install pdfplumber

In [ ]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text += page.extract_text() + "\n"
        return text
    except Exception as e:
        return f"Error extracting text: {e}"

# Replace 'your_document.pdf' with the actual path to your PDF file
pdf_text = extract_text_from_pdf('your_document.pdf')

print("--- Extracted Text (first 500 characters) ---")
print(pdf_text[:500])
print("\n--- End of Extracted Text Preview ---")

## Step 2: Load the Fintech Credit Dataset

Now, let's load the `fintech_behavior_credit_dataset.csv` into a pandas DataFrame. This will allow us to perform operations using this data.

In [ ]:
import pandas as pd

csv_file_path = '/content/fintech_behavior_credit_dataset.csv'

try:
    df = pd.read_csv(csv_file_path)
    print("Fintech Credit Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please ensure it's uploaded correctly.")
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")

## Step 3: Combine and Operate on Data

Now that you have the text extracted from the PDF and the fintech credit dataset loaded, you can perform various operations. The next steps depend heavily on:

1.  **The content of your PDF document**: What kind of information does it contain that you want to link to your credit data?
2.  **The specific operations you want to perform**: Are you trying to match entities, extract keywords, categorize text, or enrich your dataset based on the PDF content?

**Please tell me what specific operations you'd like to perform with the extracted PDF text and the credit dataset, and I can help you generate the relevant code!**

For example, you might want to:
*   Extract specific entities (e.g., company names, dates, financial figures) from the PDF text and join them with the credit dataset.
*   Categorize customers based on information in the PDF and analyze their credit behavior.
*   Use natural language processing (NLP) techniques on the PDF text to create new features for your credit model.

## Step 4: Exploratory Data Analysis (EDA) on Fintech Credit Dataset

Let's start by understanding the structure and basic statistics of our `df` DataFrame. This will help us identify potential issues like missing values, incorrect data types, or outliers.

In [ ]:
# Display basic information about the DataFrame, including data types and non-null counts
print("--- DataFrame Information ---")
df.info()

In [ ]:
# Display descriptive statistics for numerical columns
print("\n--- Descriptive Statistics for Numerical Columns ---")
display(df.describe())

In [ ]:
# Display value counts for some categorical columns to understand their distribution
print("\n--- Value Counts for 'gender' ---")
print(df['gender'].value_counts())

print("\n--- Value Counts for 'major' ---")
print(df['major'].value_counts())

print("\n--- Value Counts for 'employment_type' ---")
print(df['employment_type'].value_counts())

After reviewing these initial statistics, we can decide on further steps such as handling missing values, encoding categorical variables, or visualizing specific relationships. If you upload your PDF and tell me what information you want to extract from it, we can look into combining that with this dataset.

## Step 5: Data Cleaning

Based on our initial EDA, the `year_in_school` column has missing values. Let's address this by filling them with the mode, as it's a categorical feature. We will also check data types and ensure consistency.

In [ ]:
# Check for missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

# Fill missing values in 'year_in_school' with the mode
# The mode is suitable for categorical data
mode_year_in_school = df['year_in_school'].mode()[0]
df['year_in_school'] = df['year_in_school'].fillna(mode_year_in_school)

print("\n--- Missing Values After Cleaning ---")
print(df.isnull().sum())

print("\n--- Data types after cleaning ---")
df.info()

Now that we've handled the missing values in `year_in_school`, our dataset is cleaner. Next, we can consider further feature engineering, such as creating new features from existing ones, or preparing the data for modeling.

## Step 6: Feature Engineering

Feature engineering involves creating new features from existing data to improve the performance of machine learning models or to gain deeper insights. Let's create a few new features that could be relevant for a credit dataset.

In [ ]:
# Calculate 'spending_to_income_ratio'
df['spending_to_income_ratio'] = df['total_spending'] / df['monthly_income']

# Calculate 'disposable_income' (monthly_income - total_spending)
df['disposable_income'] = df['monthly_income'] - df['total_spending']

# Create a binary feature for 'has_financial_aid'
df['has_financial_aid'] = (df['financial_aid'] > 0).astype(int)

# Create a 'total_non_essential_spending' feature by summing entertainment, personal_care, and technology
df['total_non_essential_spending'] = df['entertainment'] + df['personal_care'] + df['technology']

print("--- New Features Created ---")
display(df[['monthly_income', 'total_spending', 'spending_to_income_ratio', 'disposable_income', 'has_financial_aid', 'total_non_essential_spending']].head())

In [ ]:
def create_labels(df):
    # Personality
    def personality(row):
        if row["total_spending"] > row["monthly_income"]:
            return "Impulsive"
        elif row["total_spending"] < 0.5 * row["monthly_income"]:
            return "Saver"
        else:
            return "Planner"

    # Risk
    def risk(row):
        if row["late_payment_ratio"] > 0.4:
            return "High"
        elif row["late_payment_ratio"] > 0.2:
            return "Medium"
        else:
            return "Low"

    df["personality"] = df.apply(personality, axis=1)
    df["risk"] = df.apply(risk, axis=1)

    return df

df = create_labels(df)


In [ ]:
cat_cols = ["gender", "year_in_school", "major", "employment_type", "preferred_payment_method"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

# Encode target labels
personality_encoder = LabelEncoder()
risk_encoder = LabelEncoder()

df["personality"] = personality_encoder.fit_transform(df["personality"])
df["risk"] = risk_encoder.fit_transform(df["risk"])

# ---------------- FEATURES ----------------
X = df.drop(["credit_score", "personality", "risk"], axis=1)

y_credit = df["credit_score"]
y_personality = df["personality"]
y_risk = df["risk"]


These new features provide a different perspective on the financial behavior of the individuals in the dataset. We can continue to create more features or move on to other steps like data visualization or model building.

## Step 7: Data Splitting

Now that we have cleaned and engineered features, we need to split our data into features (X) and the target variable (y), and then into training and testing sets. This is a crucial step for preparing the data for machine learning models. We'll use `credit_score` as our target variable. We also need to handle categorical features by converting them into numerical representations using one-hot encoding.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define features (X) and target (y)
X = df.drop('credit_score', axis=1)
y = df['credit_score']

# The original categorical columns were LabelEncoded in a previous step,
# turning them into 'int64' type. This caused select_dtypes(include=['object'])
# to return an empty list. To ensure OneHotEncoder processes these,
# we explicitly list them and then derive numerical features.
original_categorical_columns = ['gender', 'year_in_school', 'major', 'employment_type', 'preferred_payment_method']

categorical_features = [col for col in original_categorical_columns if col in X.columns]
numerical_features = [col for col in X.columns if col not in categorical_features]

# Create a column transformer for one-hot encoding categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Apply preprocessing and split the data
X_processed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding using the ColumnTransformer's method
# This method correctly combines names from all transformers (passthrough and one-hot).
feature_names = preprocessor.get_feature_names_out()

# Convert X_processed back to a DataFrame for easier inspection (optional, but good for understanding)
X_processed_df = pd.DataFrame(X_processed, columns=feature_names)

# Split the preprocessed data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_processed_df, y, test_size=0.2, random_state=42)

print("--- Data Splitting Complete ---")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

display(X_train.head())

We have successfully split the data into training and testing sets, and applied one-hot encoding to our categorical features. This prepares our data for model training. Now we can proceed with training a machine learning model.

## Step 8: Train the Model

Now that our data is prepared, we can train a machine learning model. Since `credit_score` is a numerical target, this is a regression problem. A `RandomForestRegressor` is a good choice for its robustness and performance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize the Random Forest Regressor model
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
print("--- Training the RandomForestRegressor ---")
model.fit(X_train, y_train)
print("Model training complete!")

With the model trained, let's make predictions on the test set and evaluate its performance using common regression metrics.

In [ ]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse**0.5 # Root Mean Squared Error
r2 = r2_score(y_test, y_pred)

print("--- Model Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2 ): {r2:.2f}")

## Step 9: Improve Model - Hyperparameter Tuning

To improve our `RandomForestRegressor` model, we can perform hyperparameter tuning. This involves systematically searching for the optimal combination of hyperparameters that yield the best performance on our validation set. We'll use `GridSearchCV` to explore a range of hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2'], # Number of features to consider when looking for the best split
    'max_depth': [10, 20, None], # Maximum depth of the tree
    'min_samples_split': [2, 5] # Minimum number of samples required to split an internal node
}

# Initialize GridSearchCV
# We'll use the RandomForestRegressor model initialized previously
grid_search = GridSearchCV(estimator=model, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='r2')

print("--- Starting Hyperparameter Tuning ---")
grid_search.fit(X_train, y_train)
print("--- Hyperparameter Tuning Complete ---")

# Best parameters and best score
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best R-squared score: {grid_search.best_score_:.2f}")

# Get the best model
best_model = grid_search.best_estimator_

# Evaluate the best model on the test set
y_pred_tuned = best_model.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = mse_tuned**0.5
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\n--- Tuned Model Evaluation ---")
print(f"Mean Absolute Error (MAE): {mae_tuned:.2f}")
print(f"Mean Squared Error (MSE): {mse_tuned:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse_tuned:.2f}")
print(f"R-squared (R2 ): {r2_tuned:.2f}")

In [ ]:
# 1. Credit Score Model (Regression)
credit_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    objective="reg:squarederror"
)

# 2. Personality Model (Classification)
personality_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    use_label_encoder=False,
    eval_metric="mlogloss"
)

# 3. Risk Model (Classification)
risk_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    use_label_encoder=False,
    eval_metric="mlogloss"
)



In [ ]:
X = df.drop(["credit_score", "personality", "risk"], axis=1)

y_credit = df["credit_score"]
y_personality = df["personality"]
y_risk = df["risk"]


In [ ]:
X_train, X_test, y_credit_train, y_credit_test = train_test_split(X, y_credit, test_size=0.2, random_state=42)

_, _, y_personality_train, y_personality_test = train_test_split(X, y_personality, test_size=0.2, random_state=42)

_, _, y_risk_train, y_risk_test = train_test_split(X, y_risk, test_size=0.2, random_state=42)


In [ ]:
credit_model.fit(X_train, y_credit_train)
personality_model.fit(X_train, y_personality_train)
risk_model.fit(X_train, y_risk_train)

print("\n✅ All models trained")


In [ ]:
# Credit
y_pred_credit = credit_model.predict(X_test)
print("\n📊 Credit Model:")
print("MAE:", mean_absolute_error(y_credit_test, y_pred_credit))
print("RMSE:", np.sqrt(mean_squared_error(y_credit_test, y_pred_credit)))
print("R2:", r2_score(y_credit_test, y_pred_credit))

# Personality
y_pred_personality = personality_model.predict(X_test)
print("\n🧠 Personality Model:")
print("Accuracy:", accuracy_score(y_personality_test, y_pred_personality))
print(classification_report(y_personality_test, y_pred_personality))

# Risk
y_pred_risk = risk_model.predict(X_test)
print("\n⚠️ Risk Model:")
print("Accuracy:", accuracy_score(y_risk_test, y_pred_risk))
print(classification_report(y_risk_test, y_pred_risk))


In [ ]:
import os

# Create the 'models' directory if it doesn't exist
os.makedirs('models', exist_ok=True)

joblib.dump(credit_model, "models/credit_model.pkl")
joblib.dump(personality_model, "models/personality_model.pkl")
joblib.dump(risk_model, "models/risk_model.pkl")

joblib.dump(encoders, "models/encoders.pkl")
joblib.dump(personality_encoder, "models/personality_encoder.pkl")
joblib.dump(risk_encoder, "models/risk_encoder.pkl")

We have now performed hyperparameter tuning and evaluated the best model found. This should give us an improved model performance compared to the initial default parameters.

## Step 10: Backend API with Flask

To integrate our models with a frontend, we'll create a simple Flask API. This API will expose endpoints where the frontend can send user data and receive predictions.

First, let's install Flask.

In [ ]:
pip install Flask

Now, let's set up the Flask application. This will involve loading our saved models and encoders, and defining a preprocessing function that matches the one used during training. We'll then create endpoints for each prediction task.

In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the models and encoders
credit_model = joblib.load('models/credit_model.pkl')
personality_model = joblib.load('models/personality_model.pkl')
risk_model = joblib.load('models/risk_model.pkl')
encoders = joblib.load('models/encoders.pkl') # For original categorical features
personality_encoder = joblib.load('models/personality_encoder.pkl')
risk_encoder = joblib.load('models/risk_encoder.pkl')

# Define the preprocessing logic used during training
def preprocess_input_data(data):
    # Convert input data to DataFrame
    df_input = pd.DataFrame([data])

    # Apply Label Encoding for original categorical columns
    for col, le in encoders.items():
        if col in df_input.columns:
            # Handle unseen categories by transforming known, and marking unknown
            df_input[col] = df_input[col].map(lambda s: s if s in le.classes_ else le.classes_[0]) # Map unseen to first class
            df_input[col] = le.transform(df_input[col])

    # Recreate engineered features (as done in Step 6)
    df_input['spending_to_income_ratio'] = df_input['total_spending'] / df_input['monthly_income']
    df_input['disposable_income'] = df_input['monthly_income'] - df_input['total_spending']
    df_input['has_financial_aid'] = (df_input['financial_aid'] > 0).astype(int)
    df_input['total_non_essential_spending'] = df_input['entertainment'] + df_input['personal_care'] + df_input['technology']

    # Recreate personality and risk labels (for consistent feature set if needed by models)
    def personality_func(row):
        if row["total_spending"] > row["monthly_income"]:
            return "Impulsive"
        elif row["total_spending"] < 0.5 * row["monthly_income"]:
            return "Saver"
        else:
            return "Planner"

    def risk_func(row):
        if row["late_payment_ratio"] > 0.4:
            return "High"
        elif row["late_payment_ratio"] > 0.2:
            return "Medium"
        else:
            return "Low"

    df_input["personality_label"] = df_input.apply(personality_func, axis=1)
    df_input["risk_label"] = df_input.apply(risk_func, axis=1)

    # Encode these newly created labels using their respective encoders
    df_input["personality_label"] = personality_encoder.transform(df_input["personality_label"])
    df_input["risk_label"] = risk_encoder.transform(df_input["risk_label"])

    # The XGBoost models were trained on 'X' which includes personality and risk as features.
    # Ensure the input DataFrame has all the columns X was trained on, in the correct order.
    # The 'X' in the kernel state had 27 columns after feature engineering and LabelEncoding.
    # It did *not* include 'credit_score', 'personality', and 'risk' from the original df.
    # The names 'personality_label' and 'risk_label' are used here to avoid conflict with `y_personality` and `y_risk` targets
    # and to indicate they are features derived from the input for the credit score model.

    # Ensure columns match the training features (X from cell LAXFxvp_imB0)
    # This part requires careful alignment with the X used for training.
    # Assuming X had `personality` and `risk` as features from the previous `df` transformations.

    # Let's get the exact columns used for training X from the existing `X` variable in the kernel.
    # In a real deployment, you'd save these column names.

    # For demonstration, I will assume the columns should be all columns from df_input EXCEPT
    # the original 'personality' and 'risk' which are targets, and the new 'personality_label' and 'risk_label'
    # are the ones to be used as features.

    # Based on the kernel state, `X` (cell LAXFxvp_imB0) was `df.drop(['credit_score', 'personality', 'risk'], axis=1)`.
    # So we need to ensure the input DataFrame for prediction has these columns, and `personality` and `risk`
    # are the *encoded* versions of `personality_label` and `risk_label`.

    df_input = df_input.rename(columns={'personality_label': 'personality', 'risk_label': 'risk'})

    # Drop any extra columns that weren't in the original X, but keep the ones generated
    # This is a critical step to ensure column alignment for XGBoost prediction.
    training_columns = credit_model.get_booster().feature_names # Get feature names from the trained XGBoost model

    # Add any missing columns with default value (e.g., 0) if they were in training but not in current input (shouldn't happen with full data)
    for col in training_columns:
        if col not in df_input.columns:
            df_input[col] = 0 # Or a more appropriate default/mean

    # Reorder columns to match the training data
    df_input = df_input[training_columns]

    return df_input


@app.route('/predict_credit_score', methods=['POST'])
def predict_credit_score():
    data = request.get_json(force=True)
    preprocessed_data = preprocess_input_data(data)

    prediction = credit_model.predict(preprocessed_data)
    return jsonify({'credit_score': prediction[0].item()})

@app.route('/predict_personality', methods=['POST'])
def predict_personality():
    data = request.get_json(force=True)
    preprocessed_data = preprocess_input_data(data)

    prediction_encoded = personality_model.predict(preprocessed_data)
    prediction_label = personality_encoder.inverse_transform(prediction_encoded)
    return jsonify({'personality': prediction_label[0]})

@app.route('/predict_risk', methods=['POST'])
def predict_risk():
    data = request.get_json(force=True)
    preprocessed_data = preprocess_input_data(data)

    prediction_encoded = risk_model.predict(preprocessed_data)
    prediction_label = risk_encoder.inverse_transform(prediction_encoded)
    return jsonify({'risk': prediction_label[0]})


To run this Flask app in Colab, we need to use `ngrok` to expose the local server to the internet. This will give you a public URL that your frontend can access. Remember that this URL is temporary and will change if the Colab runtime restarts.

In [ ]:
pip install flask-ngrok


In [ ]:
from flask_ngrok import run_with_ngrok

run_with_ngrok(app)  # Starts ngrok when app is run

print("Running Flask app with ngrok...")
app.run()

Once the Flask app is running, you will see a public `ngrok` URL. You can then make `POST` requests to `YOUR_NGROK_URL/predict_credit_score`, `YOUR_NGROK_URL/predict_personality`, or `YOUR_NGROK_URL/predict_risk` with a JSON payload containing the user's data.

**Example POST Request Body (JSON):**

```json
{
    "age": 25,
    "gender": "Female",
    "year_in_school": "2nd",
    "major": "Science",
    "employment_type": "Part-time",
    "monthly_income": 2500,
    "financial_aid": 500,
    "tuition": 1000,
    "housing": 600,
    "food": 300,
    "transportation": 100,
    "books_supplies": 50,
    "entertainment": 80,
    "personal_care": 30,
    "technology": 20,
    "health_wellness": 40,
    "miscellaneous": 20,
    "preferred_payment_method": "Credit Card",
    "savings": 10000,
    "monthly_transactions": 15,
    "avg_transaction_value": 150,
    "late_payment_ratio": 0.1,
    "total_spending": 2300
}
```

The API will return a JSON response with the prediction.

## Ngrok Authentication (Optional but Recommended)

To ensure `ngrok` works reliably, especially if you encounter `ConnectionRefusedError` or `ngrok` tunnels fail to establish, it's recommended to set an authentication token. You can get one from the `ngrok` dashboard after signing up.

Add your `ngrok` auth token to Colab secrets (under the "🔑" icon in the left panel) with the name `NGROK_AUTH_TOKEN`.

In [ ]:
# Import userdata for secrets
from google.colab import userdata

# Install pyngrok if not already installed
try:
    from pyngrok import ngrok
except ImportError:
    !pip install pyngrok
    from pyngrok import ngrok

# Set ngrok auth token if available in Colab secrets
# This helps prevent connection issues
NGROK_AUTH_TOKEN = None # Initialize to None

try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except Exception as e:
    # If the secret is not found, NGROK_AUTH_TOKEN remains None
    print(f"Error retrieving NGROK_AUTH_TOKEN from secrets: {e}. It might not be set.")

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("Ngrok authentication token set.")
else:
    print("Ngrok authentication token not found in secrets. Continuing without it (may lead to connection issues).")

Now, let's try running the Flask app again. Remember to run all cells from the beginning after a runtime restart to ensure all variables and models are loaded correctly.